# FSDH Databricks Python Sample
*Note: This notebook is a work in progress*

This notebook will use Python, but Databricks supports programming in SQL, Scala, and R as well.

## Connecting to storage
### Option 1: Using Blob storage
To read a file in Databricks, you can use the ABFS (Azure Blob File System). For more information on Azure Blob Storage, see: https://learn.microsoft.com/en-us/azure/storage/blobs/storage-blobs-introduction.


In [0]:
abfss = spark.conf.get('abfss_uri')
dbutils.fs.ls(abfss)
df = spark.read.option("header","true").csv(abfss + '/fsdh-sample.csv')
df.show(5)

+----+--------------+---+---------+----------+
|Name|           Sex|Age|Height_in|Weight_lbs|
+----+--------------+---+---------+----------+
|Alex|"       ""M"""| 41|       74|       170|
|Bert|"       ""M"""| 42|       68|       166|
|Carl|"       ""M"""| 32|       70|       155|
|Dave|"       ""M"""| 39|       72|       167|
|Elly|"       ""F"""| 30|       66|       124|
+----+--------------+---+---------+----------+
only showing top 5 rows


### Option 2: Mount FSDH storage using a storage key
You can also leverage Azure Key Vault to mount storage and access files through there.

In [0]:
if any(mount.mountPoint == "/mnt/fsdh" for mount in dbutils.fs.mounts()):
        dbutils.fs.unmount("/mnt/fsdh")

dbutils.fs.mount(
  source = spark.conf.get('wasbs_uri'),
  mount_point = "/mnt/fsdh",
  extra_configs = {'fs.azure.account.key.' + spark.conf.get('az_storage_name') +'.blob.core.windows.net':dbutils.secrets.get(scope = "datahub", key = "storage-key")})

dbutils.fs.ls('/mnt/fsdh')
df = spark.read.option("header","true").csv('/mnt/fsdh/fsdh-sample.csv')
df.show(5);

/mnt/fsdh has been unmounted.
+----+----------+-----+-----------+------------+
|Name|       Sex|  Age|Height (in)|Weight (lbs)|
+----+----------+-----+-----------+------------+
|Alex|       "M"|   41|         74|         170|
|Bert|       "M"|   42|         68|         166|
|Carl|       "M"|   32|         70|         155|
|Dave|       "M"|   39|         72|         167|
|Elly|       "F"|   30|         66|         124|
+----+----------+-----+-----------+------------+
only showing top 5 rows


## Connecting to a PostgreSQL database
### Option 1: Using psycopg2 (for reading and writing)
#### Step 1: Install packages

In [0]:
%pip install psycopg2

#### Step 2: Set up connection details

In [0]:
HOST="my_host"
DATABASE="my_database"
USER="my_user"
PASSWORD="my_password"

#### Step 3: Connect to database

In [0]:
import psycopg2
from psycopg2 import sql

conn = psycopg2.connect(
    host=HOST,
    database=DATABASE,
    user=USER,
    password=PASSWORD
)
cursor = conn.cursor()

#### Step 4: Sample operations

In [0]:
# Creating a table
create_table_query = """
CREATE TABLE IF NOT EXISTS celestial_bodies (
    id SERIAL PRIMARY KEY,
    name VARCHAR(100),
    body_type VARCHAR(50),
    mean_radius_km NUMERIC,
    mass_kg NUMERIC,
    distance_from_sun_km NUMERIC
);
"""
cursor.execute(create_table_query)

In [0]:
# Inserting data
dummy_data = [
    ('Mercury', 'Planet', 2439.7, 3.3011e23, 57909227),
    ('Venus', 'Planet', 6051.8, 4.8675e24, 108209475),
    ('Earth', 'Planet', 6371.0, 5.97237e24, 149598262),
    ('Mars', 'Planet', 3389.5, 6.4171e23, 227943824),
    ('Jupiter', 'Planet', 69911, 1.8982e27, 778340821),
    ('Europa', 'Moon', 1560.8, 4.7998e22, 670900000), 
    ('Ganymede', 'Moon', 2634.1, 1.4819e23, 670900000),
    ('Ceres', 'Dwarf Planet', 473, 9.3835e20, 413700000),
    ('Pluto', 'Dwarf Planet', 1188.3, 1.303e22, 5906440628)
]

insert_query = """
INSERT INTO celestial_bodies (name, body_type, mean_radius_km, mass_kg, distance_from_sun_km) VALUES (%s, %s, %s, %s, %s);
"""
cursor.executemany(insert_query, dummy_data)
conn.commit()

In [0]:
# Retrieving data
select_query = "SELECT * FROM celestial_bodies;"
cursor.execute(select_query)

rows = cursor.fetchall()
for row in rows:
    print(row)

In [0]:
# Displaying data
import pandas as pd
df = pd.read_sql_query(select_query, conn)
display(df)

#### Step 5: Close the connection

In [0]:
conn.close()

### Option 2: Spark on Databricks (read only)
#### Step 1: Set up connection details

In [0]:
HOST="my_host"
DATABASE="my_database"
USER="my_user"
PASSWORD="my_password"

#### Step 2: Read from the database

In [0]:
url = f"jdbc:postgresql://{HOST}:{5432}/{DATABASE}"
driver = "org.postgresql.Driver"

remote_table = (spark.read
    .format("jdbc")
    .option("driver", driver)
    .option("url", url)
    .option("dbtable", "celestial_bodies")
    .option("user", USER)
    .option("password", PASSWORD)
    .load()
)

#### Step 3: Display the results

In [0]:
display(remote_table)

## Further resources
For more help with Databricks, consult the [Resources section](https://poc.fsdh-dhsf.science.cloud-nuage.canada.ca/resources/) of the Federal Science DataHub.